In [1]:
pip install neo4j


In [2]:
from neo4j import GraphDatabase

In [18]:
uri = "YOUR_NEO4J_URI"
username = "neo4j"  # Default username for Aura
password = "YOUR_NEO4J_PASSWORD"
driver = GraphDatabase.driver(uri=uri, auth=(username, password))

with driver.session() as session:
    result = session.run("RETURN 'Connected to AuraDB 🎉' AS message")
    print(result.single()["message"])

Connected to AuraDB 🎉


In [19]:
import json
import math

# Define a function to filter out NaN or None values from your data
def clean_entry(entry):
    if entry.get('symbol') is None or math.isnan(entry.get('symbol', float('nan'))):
        entry['symbol'] = "default_value"  # Replace with an appropriate default value, or skip it
    return entry

# Assuming you're reading a file with JSON lines
with open('/content/drive/MyDrive/Major Project/finetune_data (step 4 result).jsonl', 'r') as f:
    for line in f:
        entry = json.loads(line)

        # Clean the entry to handle NaN values
        entry = clean_entry(entry)

        # Now execute the query with the cleaned entry
        session.write_transaction(create_graph, entry)

<ipython-input-19-21d4e57f7196>:19: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_graph, entry)


SessionError: Session closed

In [20]:
import json

def create_graph(tx, entry):
    gene_symbol = entry["Gene.symbol"]
    gene_title = entry["Gene.title"]
    regulation = entry["Regulation"]


    query = """
    MERGE (g:Gene {symbol: $gene_symbol})
    SET g.title = $gene_title



    MERGE (g)-[:INVOLVED_IN {regulation: $regulation}]->(p)
    """

    tx.run(query, gene_symbol=gene_symbol, gene_title=gene_title,
           regulation=regulation)

# Read your JSON
with open("/content/drive/MyDrive/Major Project/finetune_data (step 4 result).jsonl", "r", encoding="utf-8") as f:
    with driver.session() as session:
        for line in f:
            entry = json.loads(line)
            session.write_transaction(create_graph, entry)


<ipython-input-20-76ac0a44a73a>:26: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_graph, entry)


ClientError: {code: Neo.ClientError.Statement.SemanticError} {message: Cannot merge the following node because of NaN property value for 'symbol': (:Gene {symbol: NaN})}